# 20260919 Dataframe Construction

Build and inspect aligned session data after local H5 conversion, R2025b EXTRACT, R2021b ActSort/manualActSort labels, and curated-neuron import. Plotting lives in `20260919_visualize.ipynb`.


CaImAn co-registration is optional. This notebook builds the session dataframe from `manifest_with_cells.csv`; if `preprocess_out/coregistration/...` does not exist yet, continue normally.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import (
    cue_timestamp_overrides_path,
    default_aligned_session_path,
    default_cue_events_path,
    default_trials_path,
)
from preprocess_functions import boundary_tuning, plot, roi, segment


In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"
if not MANIFEST.exists():
    MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
RECORDING_ID = None
FPS = 30.0

# If cue_ts only gives cue onset times, set this to the known cue duration.
# If None, cue_end_utc is the next cue timestamp, and the final cue has no end.
CUE_DURATION_S = None


In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "trial_type": record.trial_type,
        "aligned_csv": str(default_aligned_session_path(OUTPUT_ROOT, record)),
        "cell_csv": str(record.cell_csv) if record.cell_csv else None,
    }
    for record in records
])
records_df


In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")

record = choose_record(records, RECORDING_ID)
aligned_csv = default_aligned_session_path(OUTPUT_ROOT, record)
print("recording_id:", record.recording_id)
print("aligned CSV:", aligned_csv)


## Upstream MATLAB Check

Before building aligned sessions, the expected order is: local H5 conversion, R2025b EXTRACT, R2021b ActSort/manualActSort labels, then curated-neuron import.

In [ ]:
print("Expected upstream files for this recording:")
print("local H5:", OUTPUT_ROOT / record.recording_id / "neural" / f"{record.recording_id}_miniscope.h5")
print("R2025b EXTRACT:", OUTPUT_ROOT / record.recording_id / "matlab" / f"{record.recording_id}_precomputed_output.mat")
print("R2021b labels:", OUTPUT_ROOT / record.recording_id / "matlab" / f"{record.recording_id}_precomputed_output_LABELS.mat")
print("curated cell CSV:", record.cell_csv)


## Build One Aligned CSV

Run this command when the aligned CSV does not exist yet. Keep `--only` while checking one recording.

In [ ]:
print(
    "python scripts/build_aligned_sessions.py "
    f"{MANIFEST} --output-root {OUTPUT_ROOT} --only {record.recording_id}"
)


In [ ]:
df = pd.read_csv(aligned_csv)
print(df.shape)
display(df.head())


## Collect or Load Arena ROIs

Run this after the aligned dataframe exists. If ROI JSON files already exist, they are loaded. If they are missing, OpenCV opens the behavior-video frame so you can draw them. Save each polygon with `s`; right click removes the last point.

The dataframe step needs `arena`, `startbox_L`, and `startbox_R` so trial segmentation can create `arena_only`.

In [ ]:
ROI_NAMES = ["arena", "startbox_L", "startbox_R"]

def choose_roi_id(record, roi_names, root=PROJECT_ROOT):
    for candidate in [record.recording_id, record.session_id]:
        if candidate and any(roi.roi_json_path(candidate, name, root=root).exists() for name in roi_names):
            return candidate
    return record.recording_id

ROI_ID = choose_roi_id(record, ROI_NAMES)

def behavior_video_path(record, df):
    if "beh_vid_path" in df.columns and df["beh_vid_path"].notna().any():
        values = df["beh_vid_path"].dropna().astype(str).str.strip()
        values = values[~values.str.lower().isin(["", "nan", "none", "null"])]
        if not values.empty:
            return Path(values.iloc[0])
    if record.beh_vid is not None:
        return Path(record.beh_vid)
    raise FileNotFoundError("No behavior video path found for ROI selection.")

video_path = behavior_video_path(record, df)
print("behavior video:", video_path)

rois, roi_paths = roi.load_or_collect_named_rois(
    video_path=video_path,
    session_id=ROI_ID,
    roi_names=ROI_NAMES,
    root=PROJECT_ROOT,
    folder_name="arena_rois",
)

height, width = roi.get_video_hw(video_path)
masks = roi.build_roi_masks(rois, height, width)
arena_mask = masks["arena"]

df = roi.add_roi_features(
    df,
    rois,
    ref_x="ear_mid_x",
    ref_y="ear_mid_y",
)
df = roi.add_arena_only_column(df)

aligned_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(aligned_csv, index=False)

print("saved ROI JSONs:")
for name, path in roi_paths.items():
    print(f"  {name}: {path}")
print("updated aligned CSV:", aligned_csv)
print("mask shape:", arena_mask.shape)
display(df[["in_arena", "in_startbox_L", "in_startbox_R", "arena_only"]].head())


## Cue Timestamp Overrides

If a cue timestamp in the spreadsheet is wrong, fix it in `preprocess_out/cue_ts_overrides.csv` instead of editing the aligned dataframe by hand. Use columns `recording_id`, `cue_ts`, and `cue_ts_note`.

In [ ]:
cue_overrides_csv = cue_timestamp_overrides_path(OUTPUT_ROOT)
cue_events_csv = default_cue_events_path(OUTPUT_ROOT, record)

if not cue_overrides_csv.exists():
    pd.DataFrame(columns=segment.CUE_OVERRIDE_COLUMNS).to_csv(cue_overrides_csv, index=False)

cue_overrides = segment.load_cue_timestamp_overrides(cue_overrides_csv)

def default_cue_from_df(df, fallback=None):
    if "cue_ts" in df.columns:
        values = df["cue_ts"].dropna().astype(str).str.strip()
        values = values[~values.str.lower().isin(["", "nan", "none", "null"])]
        if not values.empty:
            return values.iloc[0]
    return fallback

default_cue_ts = default_cue_from_df(df, record.cue_ts)
cue_ts, cue_ts_source, cue_ts_note = segment.resolve_cue_timestamp(
    recording_id=record.recording_id,
    default_cue_ts=default_cue_ts,
    cue_overrides=cue_overrides,
)
cue_events = segment.parse_cue_events(cue_ts, cue_duration_s=CUE_DURATION_S)
cue_events = segment.add_cue_event_frame_columns(cue_events, df)
cue_events_csv.parent.mkdir(parents=True, exist_ok=True)
cue_events.to_csv(cue_events_csv, index=False)

print("cue override file:", cue_overrides_csv)
print("cue events file:", cue_events_csv)
print("add/edit a row like this if the cue timestamp needs correction:")
display(pd.DataFrame([{
    "recording_id": record.recording_id,
    "cue_ts": default_cue_ts or "W 2026-08-07T00:00:00Z",
    "cue_ts_note": "manual correction note",
}]))

print("resolved cue metadata:")
display(pd.Series({
    "resolved_cue_ts": cue_ts,
    "cue_ts_source": cue_ts_source,
    "cue_ts_note": cue_ts_note,
    "cue_duration_s": CUE_DURATION_S,
}))

display(cue_events)


## Boundary Tuning and Trial Segmentation

Keep these as check cells. For batch use, move the final chosen parameters into a script in `scripts/`.

In [ ]:
cell_cols = [col for col in df.columns if col.startswith("cell_")]
print("cell columns:", len(cell_cols))


In [ ]:
if arena_mask is not None and cell_cols:
    boundary_results = boundary_tuning.compute_boundary_tuning_results(
        df,
        arena_mask,
        cell_cols=cell_cols[:5],
        n_bins=20,
        n_shuffles=100,
        random_state=0,
    )
    display(boundary_results[["cell_col", "rho", "p", "q", "significant"]])


In [ ]:
trials_csv = default_trials_path(OUTPUT_ROOT, record)
cue_events_csv = default_cue_events_path(OUTPUT_ROOT, record)
required_trial_cols = {"in_startbox_L", "in_startbox_R", "arena_only"}
missing_trial_cols = required_trial_cols - set(df.columns)

if missing_trial_cols:
    print("Missing columns for trial segmentation:", sorted(missing_trial_cols))
    trials = []
    trials_df = pd.DataFrame()
else:
    trials = segment.segment_trials_gap_window(
        df,
        fps=FPS,
        max_gap_s=3.0,
        dwell_frames=1,
        min_trial_s=0.5,
        require_opposite_side=False,
    )
    trials_df = segment.trials_to_dataframe(
        trials,
        df,
        recording_id=record.recording_id,
        session_id=record.session_id,
        mouse_id=record.mouse_id,
        trial_type=record.trial_type,
        cue_ts=cue_ts,
        cue_ts_source=cue_ts_source,
        cue_ts_note=cue_ts_note,
        cue_events=cue_events,
        cue_duration_s=CUE_DURATION_S,
        fps=FPS,
    )
    trials_csv.parent.mkdir(parents=True, exist_ok=True)
    cue_events_csv.parent.mkdir(parents=True, exist_ok=True)
    cue_events.to_csv(cue_events_csv, index=False)
    trials_df.to_csv(trials_csv, index=False)
    print("wrote trials:", trials_csv)
    print("wrote cue events:", cue_events_csv)
    display(trials_df.head())
